In [1]:
from pyspark.sql import SparkSession
import getpass


username = getpass.getuser()

In [ ]:
spark = SparkSession.builder \
.config("spark.port.ui", 0) \
.config("spark.sql.warehouse.dir", f"/user/{username}/warehouse") \
.enableHiveSupport() \
.master("yarn") \
.getOrCreate()

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql import Window

raw_music_stream_df = spark.range(100000) \
.withColumn("user_id", (F.rand() * 5000).cast("int")) \
.withColumn("genre", F.expr("case when rand() < 0.3 then 'Pop' " +
                                "when rand() < 0.6 then 'Hip-Hop' " +
                                "when rand() < 0.9 then 'Rock' " +
                                "else 'Jazz' end")) \
.withColumn("song_id", (F.rand() * 100).cast("int")) \
.withColumn("listen_duration_seconds", (F.rand() * 240 + 30).cast("int"))

display(raw_music_stream_df)

#### Top 2 songs for each genre based on their Total Listen Duration

In [ ]:
total_listen_agg = raw_music_stream_df.groupBy("song_id", "genre").agg(F.sum("listen_duration_seconds").alias("total_listen_duration"))

In [ ]:
display(total_listen_agg)

In [ ]:
window_spec = Window.partitionBy("genre").orderBy(F.desc("total_listen_duration"))

In [ ]:
rank_df = total_listen_agg.withColumn("song_rank", F.dense_rank().over(window_spec))

In [ ]:
song_listening_rank_df = rank_df.filter(F.col("song_rank") <= 2).orderBy(F.asc("genre"), F.desc("total_listen_duration"))

In [ ]:
song_listening_rank_df.show()